In [1]:
import boto3
import pandas as pd
import json

s3 = boto3.client('s3')

/usr/local/lib/python3.9/site-packages/pandas/compat/__init__.py:97: UserWarning: Could not import the lzma module. Your installed Python is incomplete. Attempting to use lzma compression will result in a RuntimeError.
  warnings.warn(msg)


In [2]:
def read_json_from_s3(bucket_name, directory):
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=directory)
    if 'Contents' not in response:
        print('Invalid directory')
    json_data = []
    for res in response['Contents']:
        file_name = res['Key']
        response = s3.get_object(Bucket=bucket_name, Key=file_name)
        lines = response['Body'].read().decode('utf-8').splitlines()
        json_data.extend([json.loads(line) for line in lines])

    return json_data

In [3]:
bucket_name = 'brickstudy'
directory = 'bronze/viral/oliveyoung/2024-08-20/'

data = read_json_from_s3(bucket_name, directory)

In [4]:
transformed_data = []
for obj in data:
    brand = list(obj.keys())[0]
    transformed_entry = {
        "brand": brand,
        "query_keyword": obj[brand]['query_keyword'],
        "brand_shop_detail_url": obj[brand]['brand_shop_detail_url'],
        "category": obj[brand]['category'],
        "released_date": obj[brand]['released_date'],
        "items": obj[brand]['items']
    }
    transformed_data.append(transformed_entry)

In [6]:
df = pd.DataFrame(transformed_data)

In [7]:
def get_top_category(cat_list):
    return set([i.split('_')[0] for i in cat_list])
df['top_category'] = df['category'].apply(get_top_category)

In [8]:
target_cat = ['스킨케어','마스크팩','클렌징','선케어','메이크업','더모코스메틱','맨즈케어','헤어케어','바디케어','향수/디퓨저']
target_set = set(target_cat)
df['allowed_cat'] = df['top_category'].apply(lambda x: list(x.intersection(target_set)))

In [9]:
target_brand = df[df['allowed_cat'].apply(lambda x: len(x)>=1)]

In [10]:
target_brand.reset_index(drop=True, inplace=True)

In [11]:
target_brand = target_brand[target_brand['items'].apply(lambda x: len(x)>=1)] #930 -> 602

In [12]:
search_keyword = target_brand['brand']
search_keyword = search_keyword.append(pd.Series(['올리브영', '올영 세일']), ignore_index=True)

In [13]:
# target_brand.to_csv('target_brand.csv', index=False)
# search_keyword.to_csv('search_keyword.csv', index=False, header=None)

In [14]:
###### Product Name Preprocessing

In [15]:
items = target_brand['items'].apply(pd.Series)
items = items.stack().reset_index(level=1)
item_df = pd.merge(target_brand, items, left_index=True, right_index=True, how = 'left') #7104 items

In [16]:
item_df.drop(['category','items','top_category'], axis=1, inplace=True)
item_df.rename(columns={'allowed_cat':'category','level_1':'item',0:'event'}, inplace=True)

In [28]:
import re

def name_preprocess(item):
    #remove brand name
    item = item.split('^')[-1]
    #remove bracket
    item = re.sub(r'\(.*?\)', '', item)
    item = re.sub(r'\[.*?\]', '', item)
    item = re.sub(r'\*.*?\*', '', item)
    #remove underscore
    item = re.sub(r'(_|\(|\))', ' ', item)
    #remove metric
    #ml, g, L, p, colors, ㎖, mm, m, ea, oz, kg, 매, 종, 호, 포, 번, 입, 개입, 개, 컬러, 패치
    item = re.sub(r'\d{1,3}(\.)?\d*(mm|번)\b', '', item, flags=re.IGNORECASE)
    item = re.sub(r'\b\d+(p).*', '', item, flags=re.IGNORECASE)
    item = re.sub(r'\d{1,3}(,\d{3})*(\+|\~|\.|\,)?\d*\s?(ml|g|L|㎖|ea|oz|kg|colors?|pcs|pads|m|병|매|매입|종|호|포|입|팩|개입|개|컬러|패치).*', '', item, flags=re.IGNORECASE)
    #remove promotions
    item = re.sub(r'\d{1,3}\+\d{1,3}', '',item) #1+1
    item = re.sub(r'대용량|\bnew\b|.*?pick', '', item, flags=re.IGNORECASE)
    item = re.sub(r'((더블|트리플|리필|증정)?\s?기획|택\s?1|단독|단품|한정).*', '', item)
    #remove multiple space
    item = re.sub(r'\s+', ' ', item).strip()
    #remove punctuation
    item = re.sub(r'[^\w\s]$', '', item)
    
    return item

In [31]:
item_df['item2'] = item_df['item'].apply(name_preprocess)

In [33]:
#error handling
item_df2 = item_df.drop_duplicates(subset=['item2'])
item_df2 = item_df2.dropna(axis=0)
item_df2 = item_df2[~(item_df2['item2']=="")]
item_df2 = item_df2[~(item_df2['brand']==item_df2['item2'])]
item_df2 = item_df2[~item_df2['brand'].isin(['바른생각','SKYN', '방탄꼭지', '숨바꼭지'])]  #remove irrelative brand
item_df2 = item_df2[~item_df2['item2'].str.contains('고데기')]

In [37]:
item_df3 = item_df2.reset_index()
item_df3 = item_df3.rename(columns={"index":"brand_index","item":"original_item","item2":"item"})

In [38]:
# item_df3.to_csv('data/preprocessed_item.csv', index=False)